# Three-way defensibility QC (Mouse SVZ pseudotime LRT)

Mouse equivalent of `organoids_knoblich/UTR/260526-three_way_defensibility.ipynb`. Computes the three-way overlap between:

1. **chi-sq cell-level LRT** (`260401-...optimized_2.R`) -- `p_value_chisq`
2. **within-sample permutation LRT** (`260526-permutation_optimized_2.R`) -- `p_value_perm`
3. **pseudobulk DRIMSeq 3-bin LRT** (`260526-drimseq_pseudotime_pseudobulk.R`) -- `adj_pvalue`

Produces the Venn diagram + summary table for the paper's Section 1.1 defensibility statement. Mouse has a single trajectory (no per-lineage split).

**Conda env**: `downstr`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026/03-optimized_method_2")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
from pathlib import Path
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from matplotlib_venn import venn3, venn3_circles

    HAVE_VENN = True
except ImportError:
    HAVE_VENN = False
    print(
        "matplotlib-venn not installed; will use a 3-set summary table instead of Venn"
    )

warnings.filterwarnings("ignore")
plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 7,
        "font.family": "sans-serif",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

BASE = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_optimized"
)
H5AD_PATH = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad"
)
# Mouse has a single trajectory; reuse the per-lineage loop with pseudo-label "all".
LINEAGES = ["all"]
PADJ_THR = 0.05


def perm_csv(lin):
    return (
        BASE
        / "permutation_batch_number"
        / "dirmult_regression_permutation_optimized2_joint_df5_pt-dpt_pseudotime_cov-batch_number.csv"
    )


def pb_csv(lin):
    return BASE / "pseudobulk_pseudotime" / "drimseq_gene_all_pseudotime.csv"


SUMMARY_OUT = BASE / "three_way_defensibility.csv"

## Build symbol-ENSEMBL map and per-lineage merged table


In [ ]:
adata = ad.read_h5ad(H5AD_PATH, backed="r")
ens_to_sym = (
    adata.var[["name", "gene_name"]]
    .drop_duplicates(subset="name")
    .rename(columns={"name": "utr_name", "gene_name": "gene_id"})
)


def load_three_way(lin):
    cp = perm_csv(lin)
    pp = pb_csv(lin)
    if not cp.exists():
        print(f"{lin}: permutation CSV not found at {cp}; run run_permutation_all.sh")
        return None
    if not pp.exists():
        print(
            f"{lin}: pseudobulk CSV not found at {pp}; run run_pseudobulk_pseudotime.sh"
        )
        return None
    perm = pd.read_csv(cp)
    # Permutation CSV has both chi-sq and permutation columns
    if "padj" not in perm.columns:
        raise ValueError("permutation CSV missing 'padj' column")
    # Also compute padj_chisq from p_value_chisq for chi-sq-only ranking
    valid_chisq = (
        perm["p_value_chisq"].notna()
        & (perm["converged"] == True)
        & (perm["lrt_status"] == "ok")
    )
    padj_chisq = np.full(len(perm), np.nan)
    from statsmodels.stats.multitest import multipletests

    p_in = perm["p_value_chisq"][valid_chisq].values
    if len(p_in) > 0:
        _, padj_in, _, _ = multipletests(p_in, method="fdr_bh")
        padj_chisq[valid_chisq.values] = padj_in
    perm["padj_chisq"] = padj_chisq
    perm = perm.merge(ens_to_sym, on="utr_name", how="left")
    pb = pd.read_csv(pp).rename(
        columns={"adj_pvalue": "padj_pseudobulk", "pvalue": "p_value_pseudobulk"}
    )
    merged = perm.merge(
        pb[
            ["gene_id", "padj_pseudobulk", "p_value_pseudobulk", "delta_psi_pseudotime"]
        ],
        on="gene_id",
        how="inner",
    )
    return merged


merged = {lin: load_three_way(lin) for lin in LINEAGES}
for lin, m in merged.items():
    if m is not None:
        print(f"{lin}: {len(m)} genes with all three test results")
        print(f"  padj_chisq populated: {m['padj_chisq'].notna().sum()}")
        print(f"  padj (perm-primary) populated: {m['padj'].notna().sum()}")
        print(f"  padj_pseudobulk populated: {m['padj_pseudobulk'].notna().sum()}")

## Per-lineage three-way overlap


In [ ]:
summary_rows = []
for lin in LINEAGES:
    m = merged[lin]
    if m is None:
        continue
    sub = m.loc[
        m["padj_chisq"].notna() & m["padj"].notna() & m["padj_pseudobulk"].notna()
    ].copy()
    n = len(sub)
    sig_chisq = sub["padj_chisq"] < PADJ_THR
    sig_perm = sub["padj"] < PADJ_THR  # primary p-value = permutation if available
    sig_pb = sub["padj_pseudobulk"] < PADJ_THR
    set_chisq = set(sub.loc[sig_chisq, "utr_name"])
    set_perm = set(sub.loc[sig_perm, "utr_name"])
    set_pb = set(sub.loc[sig_pb, "utr_name"])
    summary_rows.append(
        {
            "lineage": lin,
            "n_three_way_tested": n,
            "n_sig_chisq": len(set_chisq),
            "n_sig_perm": len(set_perm),
            "n_sig_pseudobulk": len(set_pb),
            "n_all_three": len(set_chisq & set_perm & set_pb),
            "n_chisq_and_perm_not_pb": len((set_chisq & set_perm) - set_pb),
            "n_chisq_only": len(set_chisq - set_perm - set_pb),
            "n_perm_only": len(set_perm - set_chisq - set_pb),
            "n_pb_only": len(set_pb - set_chisq - set_perm),
            "n_chisq_and_pb_not_perm": len((set_chisq & set_pb) - set_perm),
            "n_perm_and_pb_not_chisq": len((set_perm & set_pb) - set_chisq),
            "_sets": (set_chisq, set_perm, set_pb),
        }
    )

summary = pd.DataFrame(
    [{k: v for k, v in r.items() if k != "_sets"} for r in summary_rows]
)
summary.to_csv(SUMMARY_OUT, index=False)
print("Wrote:", SUMMARY_OUT)
print(summary.to_string(index=False))

## Three-set Venn per lineage


In [ ]:
for row in summary_rows:
    lin = row["lineage"]
    set_chisq, set_perm, set_pb = row["_sets"]
    fig, ax = plt.subplots(figsize=(5.5, 5))
    if HAVE_VENN:
        v = venn3(
            [set_chisq, set_perm, set_pb],
            set_labels=(
                "chi-sq cell-level",
                "permutation cell-level",
                "pseudobulk DRIMSeq",
            ),
            ax=ax,
        )
        ax.set_title(
            f"Three-way defensibility Venn (mouse SVZ)\n"
            f"FDR < {PADJ_THR}; n tested in all three = {row['n_three_way_tested']:,}",
            fontsize=9,
        )
    else:
        # Fallback: text panel
        ax.text(
            0.5,
            0.5,
            f"mouse SVZ\n\nAll three: {row['n_all_three']:,}\n"
            f"chi-sq only: {row['n_chisq_only']:,}\n"
            f"perm only: {row['n_perm_only']:,}\n"
            f"pseudobulk only: {row['n_pb_only']:,}",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=9,
        )
        ax.set_axis_off()
    figpath = BASE / "pseudobulk_pseudotime" / "figures" / f"three_way_venn_mouse.pdf"
    figpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(figpath, bbox_inches="tight")
    fig.savefig(figpath.with_suffix(".png"), bbox_inches="tight")
    plt.show()
    print(f"mouse SVZ: wrote {figpath}")

## Paper-ready summary


In [ ]:
for row in summary_rows:
    n3 = row["n_all_three"]
    n_perm = row["n_sig_perm"]
    n_pb = row["n_sig_pseudobulk"]
    pct_three_of_perm = 100 * n3 / max(n_perm, 1)
    pct_three_of_pb = 100 * n3 / max(n_pb, 1)
    print(
        f"{row['lineage']}: of {n_perm:,} permutation-significant genes at FDR<{PADJ_THR},"
        f" {n3:,} ({pct_three_of_perm:.1f}%) are also chi-sq AND pseudobulk significant."
        f" Of {n_pb:,} pseudobulk-significant genes, {n3:,} ({pct_three_of_pb:.1f}%) are"
        f" also permutation AND chi-sq significant."
    )
    print(
        f"  Cell-level-only (no pseudobulk): {row['n_chisq_and_perm_not_pb']:,}"
        f" (effects at finer pseudotime resolution than 3 bins)."
    )
    print(
        f"  Pseudobulk-only (chi-sq AND perm both n.s.): {row['n_pb_only']:,}"
        f" (potential pseudobulk red flags -- inspect)."
    )